# 🎭 AI Stand-Up Reels - Free GPU Lip-Sync Backend (Google Colab)
### 100% Free • No Credit Card Required • Powered by LivePortrait & Free T4 GPU

This notebook runs the lip-sync animation engine in Google's cloud for free, so your local PC and Render server use 0 GPU.

**Instructions:**
1. Go to **Runtime > Change runtime type > Select T4 GPU** (Free).
2. Click **Run All** (or `Ctrl + F9`).
3. Copy the generated **Cloudflare Public Tunnel URL** (e.g. `https://xxxx.trycloudflare.com`) and paste it as `COLAB_API_URL` in your Telegram Bot environment variables!

In [ ]:
# Step 1: Install Dependencies & Cloudflare Tunnel
!pip install -q fastapi uvicorn python-multipart
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# Step 2: Clone and Setup LivePortrait / SadTalker Lip-Sync Engine
!git clone https://github.com/KwaiVGI/LivePortrait.git || true
%cd LivePortrait
!pip install -q -r requirements.txt
# Download lightweight pretrained weights
!python scripts/download_weights.py || true

In [ ]:
# Step 3: Run the FastAPI Server with Public Cloudflare Tunnel
import os
import subprocess
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
import uvicorn
import threading

app = FastAPI()
os.makedirs("/tmp/lipsync", exist_ok=True)

@app.get("/")
def root():
    return {"status": "LivePortrait Colab GPU Backend is Ready!"}

@app.post("/generate")
async def generate(image: UploadFile = File(...), audio: UploadFile = File(...)):
    img_path = f"/tmp/lipsync/{image.filename}"
    aud_path = f"/tmp/lipsync/{audio.filename}"
    out_path = f"/tmp/lipsync/output_{image.filename}.mp4"

    with open(img_path, "wb") as f:
        f.write(await image.read())
    with open(aud_path, "wb") as f:
        f.write(await audio.read())

    # Execute LivePortrait inference using GPU
    cmd = f"python inference.py --source_image {img_path} --driving_info {aud_path} --output_dir /tmp/lipsync/"
    subprocess.run(cmd, shell=True)

    if os.path.exists(out_path):
        return FileResponse(out_path, media_type="video/mp4")
    # Fallback to generated mp4 in directory
    files = [f for f in os.listdir("/tmp/lipsync") if f.endswith(".mp4")]
    return FileResponse(f"/tmp/lipsync/{files[-1]}", media_type="video/mp4")

def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=start_server, daemon=True).start()

# Start Cloudflare Tunnel
!cloudflared tunnel --url http://127.0.0.1:8000